In [26]:
import sys
from pathlib import Path

try:
    project_root = Path(__file__).resolve().parent.parent
except NameError:
    project_root = Path.cwd()
    while project_root != project_root.parent and not (project_root / "src" / "features.py").exists():
        project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

REPORTS_DIR = project_root / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

In [27]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

matplotlib.use("Agg")

from src.features import ANOMALY_DAYS_EMPLOYED, build_dataset

plt.style.use("seaborn-v0_8-whitegrid")

In [3]:
df = build_dataset()

In [28]:
print(f"Размер: {df.shape}")
print(f"Доля дефолтов (TARGET = 1): {df['TARGET'].mean():.3%}")
print(f"Аномалий DAYS_EMPLOYED={ANOMALY_DAYS_EMPLOYED}: {(df['EMPLOYED_ANOMALY'] == 1).sum()}")

Размер: (307511, 139)
Доля дефолтов (TARGET = 1): 8.073%
Аномалий DAYS_EMPLOYED=365243: 55374


In [17]:
missing = df.isna().mean().sort_values(ascending=False)
print("Топ 15 признаков по пропускам:")
print(missing.head(15).map(lambda v: f"{v:.1%}"))

Топ 15 признаков по пропускам:
COMMONAREA_MODE             69.9%
COMMONAREA_AVG              69.9%
COMMONAREA_MEDI             69.9%
NONLIVINGAPARTMENTS_AVG     69.4%
NONLIVINGAPARTMENTS_MEDI    69.4%
NONLIVINGAPARTMENTS_MODE    69.4%
FONDKAPREMONT_MODE          68.4%
LIVINGAPARTMENTS_MODE       68.4%
LIVINGAPARTMENTS_MEDI       68.4%
LIVINGAPARTMENTS_AVG        68.4%
FLOORSMIN_MODE              67.8%
FLOORSMIN_AVG               67.8%
FLOORSMIN_MEDI              67.8%
YEARS_BUILD_MEDI            66.5%
YEARS_BUILD_MODE            66.5%
dtype: str


In [29]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

df["AGE_BUCKET"] = pd.cut(df["AGE_YEARS"], bins=[20, 30, 40, 50, 60, 70])
df.groupby("AGE_BUCKET", observed=True)["TARGET"].mean().plot.bar(ax=axes[0], color="#3b82f6")
axes[0].set(title="Дефолтность по возрасту", ylabel="Доля дефолтов")

df["INCOME_BUCKET"] = pd.qcut(df["AMT_INCOME_TOTAL"], q=5, duplicates="drop")
df.groupby("INCOME_BUCKET", observed=True)["TARGET"].mean().plot.bar(ax=axes[1], color="#10b981")
axes[1].set(title="Дефолтность по квантилям дохода")

df.groupby("NAME_EDUCATION_TYPE")["TARGET"].mean().sort_values().plot.barh(ax=axes[2], color="#f59e0b")
axes[2].set(title="Дефолтность по образованию", xlabel="Доля дефолтов")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_target_rates.png", dpi=150)
plt.close()

In [30]:
bureau_cols = ["BUREAU_LOAN_COUNT", "BUREAU_OVERDUE_LOANS", "BUREAU_DEBT_TO_CREDIT"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, col in zip(axes, bureau_cols):
    df.groupby(df[col].isna())["TARGET"].mean()
    bucket = pd.qcut(df[col], q=4, duplicates="drop")
    df.groupby(bucket, observed=True)["TARGET"].mean().plot.bar(ax=ax, color="#8b5cf6")
    ax.set(title=f"Дефолтность по {col}")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "eda_bureau.png", dpi=150)
plt.close()